In [ ]:
import glob
import json
import pandas as pd
from google.colab import files


In [ ]:
# ==========================================
# STEP 1: SETUP CONTAINERS
# ==========================================
# We create an empty list for each of our target tables (and one temporary for demands).
list_nodes = []
list_demands = []
list_exemplars = []
list_concepts = []
list_errors = []
list_dimensions = []

# Path to your JSON files
folder_path = "/content/PYQ_analysis/*.json"
file_list = glob.glob(folder_path)

# ==========================================
# STEP 2: LOOP & EXTRACT (File by File)
# ==========================================
for file_path in file_list:
    with open(file_path, 'r') as f:
        data = json.load(f)

    # Clean the overarching metadata once per file
    topic = data.get('topic', '').replace('-', ' ').title()




    raw_subject = str(data.get('subject', '')).strip()
    if raw_subject in ['chem', 'chemistry','Chemistry']:
        subject = 'Chemistry'
    elif raw_subject in ['math', 'maths', 'Mathematics']:
        subject = 'Mathematics'
    elif raw_subject in ['phy', 'physics']:
        subject = 'Physics'
    else:
        subject = 'Unknown'

    # 1. Archetypes (Base for dna_nodes)
    if 'archetypes' in data:
        df_a = pd.json_normalize(data, record_path=['archetypes'])
        df_a['source_topic'] = topic
        df_a['subject'] = subject
        list_nodes.append(df_a)

        # 2. Exemplars (Nested inside archetypes)
        df_e = pd.json_normalize(
            data,
            record_path=['archetypes', 'exemplars'],
            meta=[['archetypes', 'archetype_name']],
            errors='ignore'
        )
        df_e['source_topic'] = topic
        df_e['subject'] = subject
        list_exemplars.append(df_e)

    # 3. Cognitive Demands (Temporary table to merge into dna_nodes)
    if 'cognitive_demand' in data:
        df_cd = pd.json_normalize(data, record_path=['cognitive_demand'])
        df_cd['source_topic'] = topic
        list_demands.append(df_cd)

    # 4. Concept Inventory
    # (Handling as a flat list of strings based on your previous Colab setup)
    concepts = data.get('concept_inventory', [])
    if concepts:
        df_c = pd.DataFrame({'concept_name': concepts})
        df_c['topic'] = topic
        df_c['subject'] = subject
        list_concepts.append(df_c)

    # 5. Error Taxonomy
    if 'error_taxonomy' in data:
        df_err = pd.json_normalize(data, record_path=['error_taxonomy'])
        df_err['topic'] = topic
        df_err['subject'] = subject
        list_errors.append(df_err)

    # 6. Classification Dimensions
    if 'classification_dimensions' in data:
        df_dim = pd.json_normalize(data, record_path=['classification_dimensions'])
        df_dim['topic'] = topic
        df_dim['subject'] = subject
        list_dimensions.append(df_dim)

# ==========================================
# STEP 3: CONSOLIDATE
# ==========================================
# Combine all lists into master dataframes
df_base_nodes = pd.concat(list_nodes, ignore_index=True)
df_all_demands = pd.concat(list_demands, ignore_index=True)
df_exemplars = pd.concat(list_exemplars, ignore_index=True)
df_concepts = pd.concat(list_concepts, ignore_index=True) if list_concepts else pd.DataFrame()
df_error_tax = pd.concat(list_errors, ignore_index=True)
df_dimensions = pd.concat(list_dimensions, ignore_index=True)

# ==========================================
# STEP 4: MASTER TOPIC INDEXING (NEW)
# ==========================================
# Grab all unique topics, sort them ignoring case, and map them to "01", "02", etc.
all_topics = pd.concat([df_base_nodes['source_topic'], df_error_tax['topic'], df_dimensions['topic']]).dropna().unique()
sorted_topics = sorted(all_topics, key=lambda x: str(x).lower())
topic_index_map = {topic: f"{i+1:02d}" for i, topic in enumerate(sorted_topics)}


def apply_smart_ids(df, topic_col, prefix, id_col_name):
    if df.empty: return df

    # 1. Sort Alphabetically by Topic
    df = df.sort_values(by=topic_col, key=lambda x: x.str.lower()).reset_index(drop=True)

    # 2. Map the Topic Index (01, 02...)
    df['t_idx'] = df[topic_col].map(topic_index_map)

    # 3. Create a local counter per topic (001, 002, 003...)
    df['local_count'] = df.groupby(topic_col).cumcount() + 1
    df['local_idx'] = df['local_count'].astype(str).str.zfill(3)

    # 4. Map the Subject to a single letter code (C, M, P)
    subj_code_map = {'Chemistry': 'C', 'Mathematics': 'M', 'Physics': 'P'}
    df['s_code'] = df['subject'].map(subj_code_map).fillna('X')

    # 5. Construct the final ID string (e.g., N_M_01_001 or N_C_05_012)
    df[id_col_name] = prefix + "_" + df['s_code'] + "_" + df['t_idx'] + "_" + df['local_idx']

    # Drop temp columns
    return df.drop(columns=['t_idx', 'local_count', 'local_idx', 's_code'])

# ==========================================
# STEP 5: CLEAN, MERGE & FORMAT TO SCHEMA
# ==========================================

# --- TABLE 1: DNA NODES ---
# Merge base nodes with cognitive demands
df_dna_nodes = pd.merge(
    df_base_nodes,
    df_all_demands[['source_topic', 'archetype_name', 'position','reasoning']],
    on=['source_topic', 'archetype_name'],
    how='left'
)

# Rename and Map Cognitive Demand
df_dna_nodes = df_dna_nodes.rename(columns={
    'archetype_name': 'node_name',
    'position': 'cognitive_demand'
})


cog_map = {
    # Version 1 — old human readable
    "Retrieval + substitution": "Direct Application",
    "retrieval + substitution": "Direct Application",
    "Setup-heavy": "Build Then Solve",
    "setup-heavy": "Build Then Solve",
    "Inversion/synthesis": "Reverse Engineer",
    "Inversion / synthesis": "Reverse Engineer",
    "inversion / synthesis": "Reverse Engineer",

    # Version 2 — snake_case
    "retrieval_substitution": "Direct Application",
    "retrieval_to_setup": "Formula with Judgment",
    "setup_heavy": "Build Then Solve",
    "setup_to_inversion": "Build and Work Backwards",
    "inversion_synthesis": "Reverse Engineer",

    # Span values — Version 1
    "Retrieval + substitution to Setup-heavy": "Direct Application to Build Then Solve",
    "Setup-heavy to Inversion/synthesis": "Build Then Solve to Reverse Engineer",
    "Setup-heavy to Inversion / synthesis": "Build Then Solve to Reverse Engineer",

    # Span values — Version 2
    "retrieval_substitution to retrieval_to_setup": "Direct Application to Formula with Judgment",
    "retrieval_to_setup to setup_heavy": "Formula with Judgment to Build Then Solve",
    "retrieval_to_setup to setup_to_inversion": "Formula with Judgment to Build and Work Backwards",
    "retrieval_substitution to setup_heavy": "Direct Application to Build Then Solve",
    "setup_heavy to setup_to_inversion": "Build Then Solve to Build and Work Backwards",
    "setup_to_inversion to inversion_synthesis": "Build and Work Backwards to Reverse Engineer",}



df_dna_nodes['cognitive_demand'] = df_dna_nodes['cognitive_demand'].map(cog_map).fillna(df_dna_nodes['cognitive_demand'])


# Generate Smart IDs: N_C_01_001
df_dna_nodes = apply_smart_ids(df_dna_nodes, 'source_topic', 'N', 'node_id')
df_dna_nodes = df_dna_nodes[['node_id', 'node_name', 'cognitive_operation', 'subject', 'source_topic', 'reasoning', 'cognitive_demand']]


# --- TABLE 2: EXEMPLARS ---
df_exemplars = df_exemplars.rename(columns={'archetypes.archetype_name': 'node_name', 'excerpt': 'excerpt_text'})

# Bring in the newly generated node_id
df_exemplars = pd.merge(
    df_exemplars, df_dna_nodes[['node_name', 'source_topic', 'node_id']],
    on=['node_name', 'source_topic'], how='left'
)


# Generate Smart IDs: EX_C_01_001
df_exemplars = apply_smart_ids(df_exemplars, 'source_topic', 'EX', 'exemplar_id')
df_exemplars = df_exemplars[['exemplar_id', 'node_id', 'excerpt_text', 'year','source_topic']]


# --- TABLE 3: CONCEPT INVENTORY ---
if not df_concepts.empty:
    df_concepts['source'] = 'pyq'
    # Generate Smart IDs: CON_C_01_001
    df_concepts = apply_smart_ids(df_concepts, 'topic', 'CON', 'concept_id')
    df_concepts = df_concepts[['concept_id', 'topic', 'subject', 'concept_name', 'source']]


# --- TABLE 4: ERROR TAXONOMY ---
# Generate Smart IDs: ERR_C_01_001
df_error_tax = apply_smart_ids(df_error_tax, 'topic', 'ERR', 'error_id')
df_error_tax = df_error_tax[['error_id', 'topic', 'subject', 'error_name', 'description', 'error_type']]


# --- TABLE 5: QUESTION DIMENSIONS ---
# Generate Smart IDs: DIM_C_01_001
df_dimensions = apply_smart_ids(df_dimensions, 'topic', 'DIM', 'dimension_id')
df_dimensions = df_dimensions[['dimension_id', 'topic', 'subject', 'dimension_name', 'low_end', 'high_end', 'diagnostic_value']]


print(f"✅ DNA Nodes Table: {len(df_dna_nodes)} rows")
print(f"✅ Exemplars Table: {len(df_exemplars)} rows")
print(f"✅ Concepts Table: {len(df_concepts)} rows")
print(f"✅ Error Taxonomy Table: {len(df_error_tax)} rows")
print(f"✅ Dimensions Table: {len(df_dimensions)} rows")

✅ DNA Nodes Table: 363 rows
✅ Exemplars Table: 1055 rows
✅ Concepts Table: 1048 rows
✅ Error Taxonomy Table: 286 rows
✅ Dimensions Table: 138 rows


In [ ]:
df_dna_nodes.sample(30)

# c = df_dna_nodes[df_dna_nodes['source_topic'] == "Electrochemistry"]
# df_oc = df_dimensions[df_dimensions['topic'] == "Electrochemistry"]
# df_oc = df_error_tax[df_error_tax['topic'] == "Electrochemistry"]
# df_oc = df_concepts[df_concepts['topic'] == "Electrochemistry"]



# df_dna_nodes['topic_id'] = df_dna_nodes['node_id'].str.split('_').str[2]

# df_dna_nodes.groupby('source_topic')['topic_id'].unique()

In [ ]:
c = df_dna_nodes[df_dna_nodes['source_topic'] == "Salt Analysis"]

In [ ]:
df_error_tax.head()

,error_id,topic,subject,error_name,description,error_type
0,ERR_M_01_001,3D Geometry,Mathematics,Parallel vs perpendicular condition swap,Student uses the dot-product-equals-zero condi...,Conceptual error
1,ERR_M_01_002,3D Geometry,Mathematics,Direction cosine normalization neglect,"In direction cosine problems, student solves t...",Procedural error
2,ERR_M_01_003,3D Geometry,Mathematics,Line-as-plane-intersection mishandling,When a line is given as the intersection of tw...,Setup error
3,ERR_M_01_004,3D Geometry,Mathematics,Non-standard symmetric form misread,Student fails to normalize lines given in form...,Setup error
4,ERR_M_01_005,3D Geometry,Mathematics,Midpoint-image confusion,"When finding the mirror image, student compute...",Conceptual error


In [ ]:
# pd.set_option('display.max_rows', None)
display(df_dimensions)

# print(df_dimensions['dimension_name'])
# df_dimensions.shape

# Quickest way to see counts
# print(df_dimensions['topic'].value_counts())

,dimension_id,topic,subject,dimension_name,low_end,high_end,diagnostic_value
0,DIM_M_01_001,3D Geometry,Mathematics,Geometric object mixing,Question involves only one type of geometric o...,Question involves heterogeneous objects (lines...,Failure at high mixing with success at low mix...
1,DIM_M_01_002,3D Geometry,Mathematics,Representation form,All geometric objects are given in a single co...,Objects are given in mixed representations req...,Failure at mixed representations indicates a g...
2,DIM_M_01_003,3D Geometry,Mathematics,Sequential operation count,Question requires a single geometric operation...,Question chains 3+ operations where each outpu...,If a student fails on multi-step chains but ha...
3,DIM_M_01_004,3D Geometry,Mathematics,Parametric indeterminacy depth,"All geometric elements (points, lines, planes)...",One or more geometric elements contain unknown...,If a student fails at high parametric indeterm...
4,DIM_M_01_005,3D Geometry,Mathematics,Answer extraction indirectness,The question asks directly for the geometric q...,The question asks for a derived expression of ...,If a student gets the geometry right but fails...
5,DIM_M_02_001,Application Of Derivatives,Mathematics,Abstraction Level of the Function,"Explicit formula provided for f(x) (e.g., 'f(x...","Only properties of f given — f'' > 0, f(a) = b...",Failure here reveals dependence on computation...
6,DIM_M_02_002,Application Of Derivatives,Mathematics,Smoothness of the Function,Function is a smooth polynomial or trigonometr...,"Function involves absolute values, piecewise d...",Failure here indicates over-reliance on formul...
7,DIM_M_02_003,Application Of Derivatives,Mathematics,Number of Conceptual Stages,Single stage — compute derivative and answer (...,"Multi-stage — recover parameters, then compute...",Failure at the high end suggests difficulty ma...
8,DIM_M_02_004,Application Of Derivatives,Mathematics,Derivative Computation Complexity,"Standard polynomial differentiation (e.g., 'f(...","Requires implicit, parametric, or logarithmic ...",Failure at the high end reveals weak procedura...
9,DIM_M_02_005,Application Of Derivatives,Mathematics,Information Explicitness,"All information given directly — function, poi...",Key information must be derived before the mai...,Failure at the high end indicates inability to...


In [ ]:
df_concepts

,concept_id,topic,subject,concept_name,source
0,CON_M_01_001,3D Geometry,Mathematics,Shortest distance between two skew lines,pyq
1,CON_M_01_002,3D Geometry,Mathematics,Equation of line — vector and Cartesian form,pyq
2,CON_M_01_003,3D Geometry,Mathematics,Line through two points in 3D,pyq
3,CON_M_01_004,3D Geometry,Mathematics,Angle bisector plane,pyq
4,CON_M_01_005,3D Geometry,Mathematics,Normal form of plane,pyq
...,...,...,...,...,...
938,CON_M_26_019,Vector Algebra,Mathematics,Section formula and its application to divisio...,pyq
939,CON_M_26_020,Vector Algebra,Mathematics,2D rotation of vectors through a given angle w...,pyq
940,CON_M_26_021,Vector Algebra,Mathematics,Linearly dependent and independent vectors,pyq
941,CON_M_26_022,Vector Algebra,Mathematics,Distinguishing a × (b × c) from (a × b) × c (n...,pyq


In [ ]:
df_error_tax.sample(10)


,error_id,topic,subject,error_name,description,error_type
0,ERR_M_01_001,3D Geometry,Mathematics,Cross-product direction sign flip,Student computes the cross product of directio...,Procedural error
1,ERR_M_01_002,3D Geometry,Mathematics,Foot-of-perpendicular on wrong object,In questions involving both a line and a plane...,Setup error
2,ERR_M_01_003,3D Geometry,Mathematics,Family-of-planes parameter misidentification,When finding a plane through the intersection ...,Procedural error
3,ERR_M_01_004,3D Geometry,Mathematics,Direction cosine normalization neglect,"In direction cosine problems, student solves t...",Procedural error
4,ERR_M_01_005,3D Geometry,Mathematics,Coplanarity determinant row construction error,Student sets up the scalar triple product dete...,Setup error
...,...,...,...,...,...,...
281,ERR_M_27_006,Vector Algebra,Mathematics,Determinant expansion sign error,When computing the scalar triple product via a...,Procedural error
282,ERR_M_27_007,Vector Algebra,Mathematics,Angle bisector section ratio confusion,When finding the point where the angle bisecto...,Setup error
283,ERR_M_27_008,Vector Algebra,Mathematics,Perpendicularity over-constraint,"When a⃗ · c⃗ = 0 is one of several conditions,...",Conceptual error
284,ERR_M_27_009,Vector Algebra,Mathematics,Cross product direction reversal,The student computes a⃗ × b⃗ instead of b⃗ × a...,Procedural error


In [ ]:
df_exemplars
df_exemplars.to_csv('df_exemplers',index=False)

In [ ]:
# @title
def get_markdown_nodes_by_topic(df, topic):
    """
    Filters the dataframe by topic and returns a markdown-formatted string
    combining node_name and cognitive_operation.
    """
    # 1. Filter the dataframe for the specified topic
    filtered_df = df[df['source_topic'] == topic].copy()

    # Optional: Clean up any accidental line breaks inside the text fields for cleaner markdown
    filtered_df['node_name'] = filtered_df['node_name'].astype(str).str.replace('\n', ' ').str.strip()
    filtered_df['cognitive_operation'] = filtered_df['cognitive_operation'].astype(str).str.replace('\n', ' ').str.strip()

    # 2. Combine the columns into the desired markdown format: "- **Node Name**: Cognitive Operation"
    markdown_series = "-" + filtered_df['node_name'] + ": " + filtered_df['cognitive_operation']

    # 3. Join the list together with double newlines (\n\n) for proper spacing in Markdown
    markdown_output = "\n\n".join(markdown_series.tolist())

    return markdown_output

# --- Example Usage ---
topic_to_search = 'Differential Equations'
markdown_result = get_markdown_nodes_by_topic(df_dna_nodes, topic_to_search)

# Print the result (this will look great when rendered in a markdown viewer or Jupyter Notebook)

print(markdown_result)

-Disguise Unmasking: The student must recognize that the DE as written is not in standard solvable form, and perform a variable or function substitution to convert it into a recognizable type before any solving begins.

-Family Reconstruction: The student must eliminate arbitrary constants from a family of curves by differentiating the curve equation the appropriate number of times and substituting back, then identify order/degree of the resulting DE.

-Functional Equation to DE: The student must extract a differential equation from a functional equation by differentiating with respect to one variable or substituting specific values, then determine properties of the solution.

-Integral-Equation to DE Conversion: The student must differentiate a given integral equation to produce a differential equation, then solve it, with the original integral equation serving as the initial condition.

-Geometric Property Reverse-Engineering: The student must translate a geometric property of a curv

In [ ]:
# @title
def get_markdown_concepts_by_topic(df, topic):
    """
    Filters the dataframe by topic and returns a markdown-formatted string
    combining node_name and cognitive_operation.
    """
    # 1. Filter the dataframe for the specified topic
    filtered_df = df[df['topic'] == topic].copy()

    # Optional: Clean up any accidental line breaks inside the text fields for cleaner markdown
    filtered_df['concept_name'] = filtered_df['concept_name'].astype(str).str.replace('\n', ' ').str.strip()

    # 2. Combine the columns into the desired markdown format: "- **Node Name**: Cognitive Operation"
    markdown_series = filtered_df['concept_name']

    # 3. Join the list together with double newlines (\n\n) for proper spacing in Markdown
    markdown_output = "\n\n".join(markdown_series.tolist())

    return markdown_output

# --- Example Usage ---
topic_to_search = 'Matrices And Determinants'
markdown_result = get_markdown_concepts_by_topic(df_concepts, topic_to_search)

# Print the result (this will look great when rendered in a markdown viewer or Jupyter Notebook)

print(markdown_result)

Extracting eigenvalues from characteristic equations or matrix polynomial identities

Computing powers of conjugated matrices through orthogonality

Expansion using cofactors

Minors and cofactors

Translating matrix constraints into combinatorial distribution problems

Enumerating matrices with entries from finite sets satisfying aggregate conditions

Reconstructing A from indirect structural information

Determining matrix entries from its action on given input vectors

Tracking exponents across nested adj, scalar, and power operations

Recognizing that det and trace are invariant under similarity (P⁻¹AP)

Adjoint of matrix

Decomposing a matrix into symmetric and skew-symmetric parts

Identifying and exploiting skew-symmetric matrices (A^T = −A)

Using eigenvalues to compute determinant (product) and trace (sum)

Extracting unknowns from matrix polynomial identities

Using characteristic equation to express A⁻¹ or high powers as linear combinations of lower powers

Summing matrix po

In [ ]:
df_dna_nodes['source_topic'].unique()

array(['3D Geometry', 'Application Of Derivatives',
       'Area Under The Curves', 'Binomial Theorem', 'Circle',
       'Complex Numbers', 'Definite Integration',
       'Differential Equations', 'Differentiation', 'Ellipse',
       'Functions', 'Hyperbola', 'Indefinite Integrals',
       'Inverse Trigonometric Functions',
       'Limits Continuity And Differentiability', 'Logarithm',
       'Matrices And Determinants', 'Parabola',
       'Permutations And Combinations', 'Probability',
       'Quadratic Equation And Inequalities', 'Sequences And Series',
       'Sets And Relations', 'Statistics',
       'Straight Lines And Pair Of Straight Lines',
       'Trigonometric Ratio And Identites', 'Vector Algebra'],
      dtype=object)

Limits Continuity And Differentiability,Area Under The Curves, Matrices And Determinants,Quadratic Equation And Inequalities

---



In [ ]:
# @title
def get_markdown_error_by_topic(df, topic):
    """
    Filters the dataframe by topic and returns a markdown-formatted string
    combining node_name and cognitive_operation.
    """
    # 1. Filter the dataframe for the specified topic
    filtered_df = df[df['topic'] == topic].copy()

    # Optional: Clean up any accidental line breaks inside the text fields for cleaner markdown
    filtered_df['error_name'] = filtered_df['error_name'].astype(str).str.replace('\n', ' ').str.strip()
    filtered_df['description'] = filtered_df['description'].astype(str).str.replace('\n', ' ').str.strip()

    # 2. Combine the columns into the desired markdown format: "- **Node Name**: Cognitive Operation"
    markdown_series = "-" + filtered_df['error_name'] + ": " + filtered_df['description']

    # 3. Join the list together with double newlines (\n\n) for proper spacing in Markdown
    markdown_output = "\n\n".join(markdown_series.tolist())

    return markdown_output

# --- Example Usage ---
topic_to_search = 'Sets And Relations'
markdown_result = get_markdown_error_by_topic(df_error_tax, topic_to_search)

# Print the result (this will look great when rendered in a markdown viewer or Jupyter Notebook)

print(markdown_result)


-Inclusion-Exclusion Sign Error: In three-set PIE problems, the student either forgets to add back |A∩B∩C| or applies the wrong sign to pairwise intersection terms, especially when the problem gives bounds rather than exact values.

-Product-Set Index Confusion: When a relation is defined on A × B with condition involving (a₁,b₁) and (a₂,b₂), the student swaps which variable binds to which set or misinterprets the four-variable condition by accidentally treating it as a two-variable problem.

-Transitivity Counterexample Oversight: The student checks transitivity only on 'obvious' triples and misses a counterexample buried in a large relation, particularly when the relation is defined by an inequality (like 2+ab > 0) where sign combinations can produce unexpected failures.

-Vacuous Reflexivity on Empty Domain: The student fails to recognize that on domains like ℝ − {0}, a relation like ab ≥ 0 is not reflexive because 0·0 = 0 ≥ 0 but the domain-dependent version may exclude certain poi